# Agents from Scratch -- Pure Python

**Building agents, tools, design patterns, and memory systems with nothing but Python and an LLM.**

uses **GPT-4o** if an OpenAI key is available, otherwise
falls back to **Gemini 2.0 Flash**.

---

| # | Section | What We Build |
|---|---------|--------------|
| 1 | What Is an Agent? | Minimal agent loop |
| 2 | Tool Definition and Calling | Tool registry, JSON-based tool dispatch |
| 3 | Single Agent with Tools | Complete tool-calling agent |
| 4 | Design Patterns | ReAct, Reflexion, Plan-and-Execute |
| 5 | Multi-Agent Systems | Parallel agents, supervisor, debate |
| 6 | Context Principles | Pruning, summarization, quarantine, offloading |
| 7 | Memory Systems | Buffer, working, episodic, semantic, procedural |

In [9]:
# -- Setup --
#!pip install -q openai google-generativeai sentence-transformers nltk

import os, json, time, textwrap
from typing import List, Dict, Any, Callable, Tuple
from concurrent.futures import ThreadPoolExecutor
from collections import deque
from datetime import datetime

try:
    from google.colab import userdata
    OPENAI_API_KEY = userdata.get('OPENAI_API_KEY') or ''
    GEMINI_API_KEY = userdata.get('GEMINI_API_KEY') or ''
except Exception:
    OPENAI_API_KEY = os.getenv('OPENAI_API_KEY', '')
    GEMINI_API_KEY = os.getenv('GEMINI_API_KEY', '')

PROVIDER = None
openai_client = None
gemini_model = None
"""
if OPENAI_API_KEY:
    try:
        import openai
        openai_client = openai.OpenAI(api_key=OPENAI_API_KEY)
        openai_client.chat.completions.create(
            model='gpt-4o', messages=[{"role": "user", "content": "OK"}], max_tokens=5)
        PROVIDER = 'openai'
        print("[OK] Using OpenAI GPT-4o")
    except Exception as e:
        print(f"[!] OpenAI unavailable: {e}")
"""
if not PROVIDER and GEMINI_API_KEY:
    try:
        import google.generativeai as genai
        genai.configure(api_key=GEMINI_API_KEY)
        gemini_model = genai.GenerativeModel('gemini-2.0-flash')
        gemini_model.generate_content("OK")
        PROVIDER = 'gemini'
        print("[OK] Using Gemini 2.0 Flash")
    except Exception as e:
        print(f"[!] Gemini unavailable: {e}")

if not PROVIDER:
    raise ValueError("No LLM available. Add OPENAI_API_KEY or GEMINI_API_KEY to Colab secrets.")

def call_llm(prompt: str, system: str = '', temperature: float = 0.7) -> str:
    if PROVIDER == 'openai':
        msgs = []
        if system: msgs.append({"role": "system", "content": system})
        msgs.append({"role": "user", "content": prompt})
        resp = openai_client.chat.completions.create(model='gpt-4o', messages=msgs, temperature=temperature)
        return resp.choices[0].message.content.strip()
    else:
        import google.generativeai as genai
        full = f"{system}\n\n{prompt}" if system else prompt
        cfg = genai.types.GenerationConfig(temperature=temperature)
        resp = gemini_model.generate_content(full, generation_config=cfg)
        return resp.text.strip()

def call_llm_messages(messages: list, temperature: float = 0.7) -> str:
    if PROVIDER == 'openai':
        resp = openai_client.chat.completions.create(model='gpt-4o', messages=messages, temperature=temperature)
        return resp.choices[0].message.content.strip()
    else:
        import google.generativeai as genai
        parts = []
        for m in messages:
            if m['role'] == 'system': parts.insert(0, f"Instructions: {m['content']}")
            else: parts.append(f"{'User' if m['role']=='user' else 'Assistant'}: {m['content']}")
        cfg = genai.types.GenerationConfig(temperature=temperature)
        resp = gemini_model.generate_content('\n\n'.join(parts) + '\nAssistant:', generation_config=cfg)
        return resp.text.strip()

print(f"Provider: {PROVIDER}")
print(call_llm('Reply with exactly: AGENTS READY', temperature=0))

ValueError: No LLM available. Add OPENAI_API_KEY or GEMINI_API_KEY to Colab secrets.

---
# Part 1 -- What Is an Agent?

An agent is an LLM in a **loop** that can perceive, reason, act, observe, and repeat.

```
while not done:
    thought = llm("What should we do next?")
    if thought says "DONE": break
    result = execute(thought)
    context += result
```

In [ ]:
# -- Part 1: Minimal Agent --
def minimal_agent(task: str, max_steps: int = 5) -> str:
    context = f"Task: {task}\n"
    for step in range(1, max_steps + 1):
        prompt = (
            f"{context}\nStep {step}: Think about what to do next. "
            f"If done, start with DONE: followed by the final answer."
        )
        response = call_llm(prompt, system="We are a step-by-step thinker. Be concise.", temperature=0.3)
        print(f"  [Step {step}] {response[:150]}...")
        context += f"\nStep {step}: {response}"
        if response.upper().startswith("DONE:"):
            return response[5:].strip()
    return response

print("MINIMAL AGENT")
print("=" * 50)
result = minimal_agent("What are the three main benefits of containerization?")
print(f"\nFinal: {result[:300]}")

MINIMAL AGENT
  [Step 1] Step 1: Identify the key benefits of containerization in software development and deployment.

1. **Portability**: Containers encapsulate applications...
  [Step 2] DONE: The three main benefits of containerization are portability, scalability, and isolation....

Final: The three main benefits of containerization are portability, scalability, and isolation.


---
# Part 2 -- Tool Definition and Calling

Tools are Python functions the agent can invoke. We need:
1. A way to **describe** tools to the LLM
2. A way to **execute** the chosen tool
3. A way to **parse** the LLM's tool-call decision (JSON)

In [ ]:
# -- Part 2: Tools --
def search_knowledge_base(query: str) -> str:
    """Search our internal knowledge base for information."""
    kb = {
        "pricing": "Standard: $29/mo, Pro: $99/mo, Enterprise: custom. 14-day trial.",
        "refund": "Full refund within 30 days. 50% within 60 days. Gold members: 90-day full refund.",
        "features": "Core: dashboards, API, collaboration. Pro: integrations, priority support, SSO.",
        "security": "SOC2 Type II. AES-256 at rest, TLS 1.3 in transit. GDPR compliant.",
    }
    for key, val in kb.items():
        if key in query.lower(): return val
    return f"No results for '{query}'. Topics: {', '.join(kb.keys())}"

def calculate(expression: str) -> str:
    """Evaluate a math expression using Python syntax."""
    try: return f"Result: {eval(expression, {'__builtins__': {}})}"
    except Exception as e: return f"Error: {e}"

def get_current_date() -> str:
    """Get today's date."""
    return f"Today: {datetime.now().strftime('%Y-%m-%d')}"

def web_search(query: str) -> str:
    """Search the web for information (simulated)."""
    return f"Web results for '{query}': Found 5 relevant articles."

class ToolRegistry:
    def __init__(self):
        self.tools: Dict[str, Dict] = {}
    def register(self, func):
        import inspect
        sig = inspect.signature(func)
        params = {n: {"type": "string"} for n in sig.parameters}
        self.tools[func.__name__] = {"function": func, "description": func.__doc__ or "", "parameters": params}
        return func
    def describe(self) -> str:
        lines = []
        for name, info in self.tools.items():
            params = ", ".join(info['parameters'].keys())
            lines.append(f"- {name}({params}): {info['description']}")
        return "\n".join(lines)
    def execute(self, name: str, args: dict) -> str:
        if name not in self.tools: return f"Error: Unknown tool '{name}'"
        try: return str(self.tools[name]["function"](**args))
        except Exception as e: return f"Error: {e}"

registry = ToolRegistry()
for fn in [search_knowledge_base, calculate, get_current_date, web_search]:
    registry.register(fn)

print("Tools registered:")
print(registry.describe())

def parse_tool_call(response: str) -> dict:
    text = response.strip().removeprefix("```json").removeprefix("```").removesuffix("```").strip()
    try: return json.loads(text)
    except: return {"tool": "none", "response": text}

Tools registered:
- search_knowledge_base(query): Search our internal knowledge base for information.
- calculate(expression): Evaluate a math expression using Python syntax.
- get_current_date(): Get today's date.
- web_search(query): Search the web for information (simulated).


In [ ]:
# -- Tool Calling Demo --
TOOL_SYS = (
    "We are a helpful assistant with tools.\n\nAvailable tools:\n{tools}\n\n"
    "To use a tool: {{\"tool\": \"name\", \"args\": {{...}}}}\n"
    "To respond directly: {{\"tool\": \"none\", \"response\": \"answer\"}}"
)

system = TOOL_SYS.format(tools=registry.describe())
for q in ["What is our Pro plan pricing?", "Calculate 99 * 12", "What date is it?"]:
    raw = call_llm(q, system=system, temperature=0)
    parsed = parse_tool_call(raw)
    tool = parsed.get("tool", "none")
    print(f"Q: {q}")
    if tool != "none":
        result = registry.execute(tool, parsed.get("args", {}))
        print(f"  Tool: {tool} -> {result}")
    else:
        print(f"  Direct: {parsed.get('response', raw)[:150]}")
    print()

Q: What is our Pro plan pricing?
  Tool: search_knowledge_base -> Standard: $29/mo, Pro: $99/mo, Enterprise: custom. 14-day trial.

Q: Calculate 99 * 12
  Tool: calculate -> Result: 1188

Q: What date is it?
  Tool: get_current_date -> Today: 2026-03-13



---
# Part 3 -- Single Agent with a Full Tool Loop

We combine the LLM, tools, and a loop into a complete agent:
1. Receive user question
2. Decide: call a tool or respond directly
3. If tool called, feed the result back and decide again
4. Repeat until the agent responds directly

In [ ]:
# -- Part 3: Single Agent with Tool Loop --
class SimpleAgent:
    def __init__(self, system_prompt: str, tool_reg: ToolRegistry):
        self.system_prompt = system_prompt
        self.registry = tool_reg
        self.tool_calls_log = []

    def run(self, user_input: str, max_turns: int = 6) -> str:
        system = (
            f"{self.system_prompt}\n\nAvailable tools:\n{self.registry.describe()}\n\n"
            f"To use a tool: {{\"tool\": \"name\", \"args\": {{...}}}}\n"
            f"When done: {{\"tool\": \"none\", \"response\": \"final answer\"}}"
        )
        messages = [{"role": "system", "content": system}, {"role": "user", "content": user_input}]
        self.tool_calls_log = []

        for turn in range(max_turns):
            raw = call_llm_messages(messages, temperature=0.3)
            parsed = parse_tool_call(raw)
            tool_name = parsed.get("tool", "none")

            if tool_name == "none" or not tool_name:
                final = parsed.get("response", raw)
                print(f"  [ANSWER] {final[:200]}")
                return final

            args = parsed.get("args", {})
            result = self.registry.execute(tool_name, args)
            self.tool_calls_log.append({"tool": tool_name, "args": args, "result": result})
            print(f"  [TOOL] {tool_name}({args}) -> {result[:100]}")
            messages.append({"role": "assistant", "content": raw})
            messages.append({"role": "user", "content": f"Tool result: {result}"})

        return "Max turns reached."

agent = SimpleAgent("We are a helpful customer service agent.", registry)
print("SINGLE AGENT DEMO")
print("=" * 50)
for q in [
    "What is the annual cost of our Pro plan?",
    "Can a customer get a refund after 45 days? What about Gold members?",
]:
    print(f"\nQ: {q}")
    agent.run(q)
    print(f"  Tool calls made: {len(agent.tool_calls_log)}")

SINGLE AGENT DEMO

Q: What is the annual cost of our Pro plan?
  [ANSWER] To assist you with the cost of the Pro plan, I will need to check our internal knowledge base for the most accurate and up-to-date information. Let me do that for you.
  Tool calls made: 0

Q: Can a customer get a refund after 45 days? What about Gold members?
  [ANSWER] To provide accurate information regarding refund policies, I will need to check the specific terms and conditions related to refunds and any special provisions for Gold members. Let me search our inte
  Tool calls made: 0


---
# Part 4 -- Agent Design Patterns

| Pattern | Loop | Best For |
|---------|------|----------|
| **ReAct** | Thought -> Action -> Observation | Tool-heavy tasks |
| **Reflexion** | Act -> Evaluate -> Reflect -> Retry | Quality-sensitive tasks |
| **Plan-and-Execute** | Plan -> Execute steps -> Replan | Complex multi-step tasks |

In [ ]:
# -- Pattern 1: ReAct (Reason + Act) --
class ReActAgent:
    def __init__(self, tool_reg: ToolRegistry):
        self.registry = tool_reg
        self.trace = []

    def run(self, question: str, max_steps: int = 5) -> str:
        tool_desc = self.registry.describe()
        context = f"Question: {question}\n"

        for step in range(1, max_steps + 1):
            # THOUGHT
            thought = call_llm(
                f"{context}\nAvailable tools:\n{tool_desc}\n\n"
                f"Step {step} THOUGHT: What do we know? What do we need? "
                f"If we can answer, say ANSWER: followed by the answer.",
                temperature=0.3)
            self.trace.append(("thought", thought))
            print(f"  [THOUGHT {step}] {thought[:150]}")

            if "ANSWER:" in thought.upper():
                idx = thought.upper().index("ANSWER:")
                return thought[idx+7:].strip()

            # ACTION
            raw = call_llm(
                f"{context}Thought: {thought}\n\nTools:\n{tool_desc}\n\n"
                f"Which tool? JSON: {{\"tool\": \"name\", \"args\": {{...}}}}",
                temperature=0)
            parsed = parse_tool_call(raw)
            tool_name = parsed.get("tool", "none")

            if tool_name == "none":
                return parsed.get("response", thought)

            args = parsed.get("args", {})
            result = self.registry.execute(tool_name, args)
            self.trace.append(("action", f"{tool_name}({args})"))
            self.trace.append(("observation", result))
            print(f"  [ACTION {step}] {tool_name}({args})")
            print(f"  [OBS {step}] {result[:120]}")
            context += f"Thought: {thought}\nAction: {tool_name}({args})\nObservation: {result}\n"

        return "Max steps."

react = ReActAgent(registry)
print("REACT AGENT")
print("=" * 50)
answer = react.run("How much would 2 years of Pro plan cost?")
print(f"\nFinal: {answer[:300]}")
print(f"Trace: {len(react.trace)} entries")

REACT AGENT
  [THOUGHT 1] To determine the cost of a 2-year Pro plan, we need to know the monthly or annual cost of the Pro plan. Once we have that information, we can calculat

Final: I'm sorry, but I don't have access to real-time data or the ability to search the web. However, I can guide you on how to find the information you need.

To determine the cost of a 2-year Pro plan, you should:

1. Visit the official website of the service or product offering the Pro plan.
2. Look fo
Trace: 1 entries


In [ ]:
# -- Pattern 2: Reflexion (Act -> Evaluate -> Reflect -> Retry) --
class ReflexionAgent:
    def __init__(self):
        self.attempts = []

    def run(self, task: str, max_attempts: int = 3) -> str:
        for attempt in range(1, max_attempts + 1):
            if attempt == 1:
                prompt = f"Task: {task}\n\nProvide a thorough answer."
            else:
                prev = self.attempts[-1]
                prompt = (
                    f"Task: {task}\n\nPrevious attempt:\n{prev['answer'][:400]}\n\n"
                    f"Evaluation: {prev['evaluation'][:200]}\n\n"
                    f"Reflection: {prev['reflection'][:200]}\n\n"
                    f"Provide an improved answer addressing all weaknesses.")

            answer = call_llm(prompt, system="We are a thorough analyst.", temperature=0.7)
            print(f"  [ATTEMPT {attempt}] {answer[:120]}...")

            evaluation = call_llm(
                f"Task: {task}\nAnswer:\n{answer[:500]}\n\n"
                f"Score 1-10: SCORE: N/10 and brief justification.", temperature=0.3)
            print(f"  [EVAL {attempt}] {evaluation[:120]}")

            score = 0
            for part in evaluation.replace(':', ' ').split():
                try:
                    if '/' in part: score = int(part.split('/')[0]); break
                except: pass

            reflection = call_llm(
                f"Answer:\n{answer[:400]}\nEval: {evaluation[:200]}\n"
                f"List 2-3 specific improvements.", temperature=0.3)
            print(f"  [REFLECT {attempt}] {reflection[:120]}")

            self.attempts.append({"answer": answer, "evaluation": evaluation,
                                  "reflection": reflection, "score": score})
            if score >= 8:
                print(f"  [PASS] Score {score}/10")
                return answer

        best = max(self.attempts, key=lambda a: a["score"])
        return best["answer"]

ref = ReflexionAgent()
print("REFLEXION AGENT")
print("=" * 50)
result = ref.run("Explain tradeoffs between SQL and NoSQL for real-time analytics.")
print(f"\nFinal ({len(ref.attempts)} attempts): {result[:300]}")

REFLEXION AGENT
  [ATTEMPT 1] When considering real-time analytics, the choice between SQL (Structured Query Language) and NoSQL (Not Only SQL) databa...
  [EVAL 1] **Score: 7/10**

**Justification:**

The response begins well by setting the context for the tradeoffs between SQL and N
  [REFLECT 1] To improve the response regarding the tradeoffs between SQL and NoSQL databases for real-time analytics, consider the fo
  [ATTEMPT 2] When exploring the tradeoffs between SQL (Structured Query Language) and NoSQL (Not Only SQL) databases for real-time an...
  [EVAL 2] Score: 7/10

Justification: The response begins by correctly identifying key factors to consider when comparing SQL and 
  [REFLECT 2] 1. **Complete the Analysis:** The response starts with a promising introduction but cuts off before fully discussing the
  [ATTEMPT 3] To effectively evaluate the tradeoffs between SQL and NoSQL databases for real-time analytics, it's crucial to assess va...
  [EVAL 3] ### SQL Databases

**Stre

In [ ]:
# -- Pattern 3: Plan-and-Execute --
class PlanExecuteAgent:
    def __init__(self, tool_reg: ToolRegistry):
        self.registry = tool_reg
        self.plan = []
        self.results = []

    def run(self, task: str) -> str:
        # PLAN
        raw = call_llm(
            f"Task: {task}\nTools: {', '.join(self.registry.tools.keys())}\n\n"
            f"Create 3-5 steps. Return JSON list: [\"step1\", \"step2\", ...]",
            temperature=0.3)
        try:
            clean = raw.strip().removeprefix("```json").removeprefix("```").removesuffix("```").strip()
            self.plan = json.loads(clean)
        except: self.plan = [raw]

        print(f"  [PLAN] {self.plan}")
        self.results = []

        # EXECUTE each step
        for i, step in enumerate(self.plan):
            ctx = "\n".join(f"Step {j+1}: {r}" for j, r in enumerate(self.results))
            raw = call_llm(
                f"Context:\n{ctx}\n\nCurrent step: {step}\nTools:\n{self.registry.describe()}\n\n"
                f"Execute. If tool needed: {{\"tool\": \"name\", \"args\": {{...}}}}\n"
                f"Otherwise just provide the result.", temperature=0.3)
            parsed = parse_tool_call(raw)
            if parsed.get("tool") and parsed["tool"] != "none":
                result = self.registry.execute(parsed["tool"], parsed.get("args", {}))
            else:
                result = parsed.get("response", raw)
            self.results.append(result)
            print(f"  [STEP {i+1}] {step[:60]} -> {result[:80]}")

        # SYNTHESIZE
        all_res = "\n".join(f"Step {i+1}: {r}" for i, r in enumerate(self.results))
        return call_llm(f"Task: {task}\n\nResults:\n{all_res}\n\nSynthesize into a clear answer.", temperature=0.5)

pe = PlanExecuteAgent(registry)
print("PLAN-AND-EXECUTE")
print("=" * 50)
result = pe.run("Compare our pricing with competitors and calculate 3-year Pro plan cost.")
print(f"\nFinal: {result[:400]}")

PLAN-AND-EXECUTE
  [PLAN] ['step1: Use web_search to find our current Pro plan pricing details.', 'step2: Use web_search to gather Pro plan pricing details of our competitors.', 'step3: Use calculate to determine the total cost of our Pro plan over 3 years.', "step4: Compare our 3-year Pro plan cost with competitors' pricing.", 'step5: Summarize the findings and highlight any pricing advantages or disadvantages.']
  [STEP 1] step1: Use web_search to find our current Pro plan pricing d -> Web results for 'current Pro plan pricing details': Found 5 relevant articles.
  [STEP 2] step2: Use web_search to gather Pro plan pricing details of  -> Web results for 'Pro plan pricing details of competitors': Found 5 relevant arti
  [STEP 3] step3: Use calculate to determine the total cost of our Pro  -> To determine the total cost of our Pro plan over 3 years, I need to know the cur
  [STEP 4] step4: Compare our 3-year Pro plan cost with competitors' pr -> To proceed with the comparison of our 3-y

---
# Part 5 -- Multi-Agent Systems

Split work across multiple agents, each with isolated context:
- **Parallel** -- each handles one aspect independently (quarantine)
- **Supervisor** -- delegates to specialists, synthesizes results
- **Debate** -- two agents argue, a judge decides

In [ ]:
# -- Part 5: Multi-Agent --
class Agent:
    def __init__(self, name: str, system_prompt: str):
        self.name = name
        self.system = system_prompt
        self.history = []

    def __call__(self, msg: str) -> str:
        self.history.append({"role": "user", "content": msg})
        messages = [{"role": "system", "content": self.system}] + self.history
        resp = call_llm_messages(messages, temperature=0.7)
        self.history.append({"role": "assistant", "content": resp})
        return resp

# -- Parallel (Quarantined) --
print("PARALLEL AGENTS (QUARANTINED)")
print("=" * 50)
topic = "serverless computing platforms"
agents = {
    "market": Agent("Market", "We are a market analyst. Focus ONLY on market trends. 3 sentences max."),
    "technical": Agent("Tech", "We are a tech analyst. Focus ONLY on architecture. 3 sentences max."),
    "competition": Agent("Comp", "We are a competitive analyst. Focus ONLY on key players. 3 sentences max."),
}

def run_aspect(pair):
    name, ag = pair
    return name, ag(f"Analyze {name} aspects of: {topic}")

with ThreadPoolExecutor(max_workers=3) as pool:
    results = dict(pool.map(run_aspect, agents.items()))

for name, text in results.items():
    print(f"\n[{name.upper()}] {text[:200]}")

# -- Supervisor --
print("\n" + "=" * 50)
print("SUPERVISOR PATTERN")
supervisor = Agent("Supervisor", "We are a lead analyst. Synthesize into an executive summary.")
combined = "\n".join(f"[{n}] {t[:300]}" for n, t in results.items())
print(supervisor(f"Synthesize about {topic}:\n{combined}")[:400])

# -- Debate --
print("\n" + "=" * 50)
print("DEBATE PATTERN")
q = "Should a startup use serverless or containers?"
pro = Agent("Pro-Serverless", "We argue FOR serverless. 3 reasons.")
con = Agent("Pro-Containers", "We argue FOR containers. 3 reasons.")

a1, a2 = pro(q), con(q)
print(f"[PRO-SERVERLESS] {a1[:200]}")
print(f"\n[PRO-CONTAINERS] {a2[:200]}")

judge = Agent("Judge", "We are an impartial judge. Evaluate both fairly.")
verdict = judge(f"FOR serverless:\n{a1[:300]}\n\nFOR containers:\n{a2[:300]}\n\nVerdict?")
print(f"\n[VERDICT] {verdict[:300]}")

PARALLEL AGENTS (QUARANTINED)

[MARKET] The market for serverless computing platforms is experiencing significant growth, driven by the increasing demand for scalable and cost-efficient cloud solutions. Tech giants like AWS, Microsoft Azure

[TECHNICAL] Serverless computing platforms abstract server management by automatically scaling and executing code in response to events, allowing developers to focus solely on writing application logic. These pla

[COMPETITION] Key players in the serverless computing platforms space include Amazon Web Services (AWS) Lambda, Microsoft Azure Functions, and Google Cloud Functions. AWS Lambda is a leader due to its early market 

SUPERVISOR PATTERN
**Executive Summary:**

The serverless computing platforms market is rapidly expanding, driven by the growing need for scalable and cost-effective cloud solutions. Major technology companies such as Amazon Web Services (AWS), Microsoft Azure, and Google Cloud are intensively broadening their serverless offer

---
# Part 6 -- Context Principles

Five techniques for controlling what the LLM sees:

| Technique | Problem It Solves |
|-----------|------------------|
| **Tool Loadout** | Too many tools confuse the LLM |
| **Context Quarantine** | Sub-task noise leaks between agents |
| **Context Pruning** | Irrelevant text wastes tokens and distracts |
| **Context Summarization** | Growing context causes drift |
| **Context Offloading** | Intermediate reasoning clutters main thread |

In [ ]:
# -- Part 6: Context Principles --

# 6A: Tool Loadout (Less-is-More) using SentenceTransformer
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

print("Loading embedding model...")
embed_model = SentenceTransformer('sentence-transformers/all-mpnet-base-v2')

tool_names = list(registry.tools.keys())
tool_descs = [f"{n}: {registry.tools[n]['description']}" for n in tool_names]
tool_embeds = embed_model.encode(tool_descs)

def select_tools(query: str, top_k: int = 2) -> list:
    """Select only the most relevant tools for a query using embeddings."""
    q_embed = embed_model.encode([query])
    sims = cosine_similarity(q_embed, tool_embeds)[0]
    top_idx = sims.argsort()[-top_k:][::-1]
    return [(tool_names[i], float(sims[i])) for i in top_idx]

print("\nTool Loadout Demo:")
for q in ["What is our pricing?", "Calculate 99*12", "Search the web for Python tutorials"]:
    selected = select_tools(q)
    print(f"  Query: '{q}'")
    print(f"  Selected: {[(n, f'{s:.3f}') for n, s in selected]}")
    print(f"  Reduction: {len(tool_names)} tools -> {len(selected)} (saved {len(tool_names)-len(selected)} from context)")
    print()

Loading embedding model...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]


Tool Loadout Demo:
  Query: 'What is our pricing?'
  Selected: [('calculate', '0.189'), ('search_knowledge_base', '0.044')]
  Reduction: 4 tools -> 2 (saved 2 from context)

  Query: 'Calculate 99*12'
  Selected: [('calculate', '0.345'), ('web_search', '0.122')]
  Reduction: 4 tools -> 2 (saved 2 from context)

  Query: 'Search the web for Python tutorials'
  Selected: [('calculate', '0.324'), ('web_search', '0.298')]
  Reduction: 4 tools -> 2 (saved 2 from context)



In [ ]:
# 6B: Context Pruning
import nltk
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

doc = (
    "AWS Lambda is serverless compute. It supports Python, Node.js, Java. "
    "Pizza is an Italian dish made with dough and cheese. "
    "Lambda scales automatically based on incoming requests. "
    "The Eiffel Tower is 330 meters tall. "
    "Lambda pricing is per-request and per-duration of execution."
)

def prune_context(question: str, document: str) -> str:
    sents = nltk.sent_tokenize(document)
    numbered = '\n'.join(f'[{i}] {s}' for i, s in enumerate(sents))
    raw = call_llm(
        f"Return ONLY the sentence indices relevant to the question as a JSON array of integers.\n\n"
        f"Question: {question}\nSentences:\n{numbered}", temperature=0)
    try:
        clean = raw.strip().removeprefix("```json").removeprefix("```").removesuffix("```").strip()
        indices = json.loads(clean)
        return ' '.join(sents[i] for i in indices if i < len(sents))
    except: return document

print("CONTEXT PRUNING")
print("=" * 50)
question = "What languages does Lambda support?"
pruned = prune_context(question, doc)
print(f"Original ({len(doc)} chars): {doc[:120]}...")
print(f"Pruned   ({len(pruned)} chars): {pruned}")
print(f"Reduction: {(1 - len(pruned)/len(doc))*100:.0f}%")


# 6C: Context Summarization
print("\n" + "=" * 50)
print("CONTEXT SUMMARIZATION")
conversation = [
    ("user", "What Python web framework should we use?"),
    ("assistant", "FastAPI for async APIs. Django for full-stack."),
    ("user", "We need async. How does FastAPI handle auth?"),
    ("assistant", "FastAPI uses OAuth2 with JWT. Very flexible."),
    ("user", "What about database support?"),
    ("assistant", "SQLAlchemy async with FastAPI. PostgreSQL recommended."),
    ("user", "How do we handle migrations?"),
    ("assistant", "Use Alembic. It integrates directly with SQLAlchemy."),
]

# Simulate growing conversation that triggers summarization
buffer = []
summary = None
MAX_BUFFER = 4

for role, content in conversation:
    buffer.append(f"{role}: {content}")
    if len(buffer) > MAX_BUFFER:
        summary = call_llm(
            f"Summarize preserving key decisions:\n" + "\n".join(buffer[:-2]),
            temperature=0.3)
        buffer = buffer[-2:]
        print(f"  [SUMMARIZED] {summary[:150]}")

print(f"\nFinal buffer ({len(buffer)} items): {buffer}")
print(f"Summary: {summary[:200] if summary else 'None'}")


# 6D: Context Offloading (Think Tool)
print("\n" + "=" * 50)
print("CONTEXT OFFLOADING (THINK TOOL)")

REFUND_POLICY = (
    "Full refund within 30 days. 50% within 60 days. None after 60 days. "
    "Gold members: 90-day full refund. Damaged items: always full refund."
)

def agent_with_think(question: str) -> dict:
    # Private reasoning -- not included in the customer-facing response
    thought = call_llm(
        f"POLICY: {REFUND_POLICY}\n\nRequest: {question}\n\n"
        f"Think through each policy rule step by step. Which rules apply? What is the correct action?",
        system="We are checking policy compliance. Be thorough.", temperature=0.3)
    print(f"  [THINK -- private] {thought[:250]}...")

    # Customer-facing answer informed by private reasoning
    answer = call_llm(
        f"Request: {question}\n\nWe have determined: {thought[:300]}\n\n"
        f"Provide a clear, professional response to the customer.",
        temperature=0.5)
    return {"answer": answer, "thought": thought}

result = agent_with_think("I want a refund for order ORD-123. I am a Gold member and it was delivered 50 days ago.")
print(f"\n  [ANSWER] {result['answer'][:300]}")

CONTEXT PRUNING
Original (275 chars): AWS Lambda is serverless compute. It supports Python, Node.js, Java. Pizza is an Italian dish made with dough and cheese...
Pruned   (34 chars): It supports Python, Node.js, Java.
Reduction: 88%

CONTEXT SUMMARIZATION
  [SUMMARIZED] FastAPI handles authentication by providing support for OAuth2, JWT tokens, and other authentication methods. It allows you to define security schemes
  [SUMMARIZED] FastAPI integrates OAuth2 with JWT for authentication, offering flexibility. For database support, it uses SQLAlchemy with asynchronous capabilities, 

Final buffer (2 items): ['user: How do we handle migrations?', 'assistant: Use Alembic. It integrates directly with SQLAlchemy.']
Summary: FastAPI integrates OAuth2 with JWT for authentication, offering flexibility. For database support, it uses SQLAlchemy with asynchronous capabilities, and PostgreSQL is recommended as the database choi

CONTEXT OFFLOADING (THINK TOOL)
  [THINK -- private] To determine the 

---
# Part 7 -- Memory Systems

Agent memory mirrors human memory:

| Type | What It Stores | Retention | Implementation |
|------|---------------|-----------|----------------|
| **Buffer** | Last N messages | Very short | `deque(maxlen=N)` |
| **Working** | Current reasoning | Current task | List (scratchpad) |
| **Episodic** | Past interactions | Long-term | Timestamped logs |
| **Semantic** | Facts and entities | Permanent | Dict of facts |
| **Procedural** | Learned strategies | Permanent | Dict of procedures |

In [ ]:
from collections import deque
from datetime import datetime
from typing import Dict, List


# ── Buffer Memory -- fixed-size sliding window ───────────────────────────────
class BufferMemory:
    def __init__(self, max_size: int = 6):
        self.buffer = deque(maxlen=max_size)

    def add(self, role: str, content: str):
        self.buffer.append({"role": role, "content": content})

    def get_messages(self) -> list:
        return list(self.buffer)

    def __len__(self):
        return len(self.buffer)


buf = BufferMemory(max_size=4)

for i in range(6):
    buf.add("user", f"Message {i+1}")
    buf.add("assistant", f"Reply {i+1}")

print(f"Buffer (max 4, added 12): {len(buf)} items")
print(f"Contents: {[m['content'] for m in buf.get_messages()]}")
print("Oldest messages automatically dropped.\n")


# ── Working Memory -- scratchpad ─────────────────────────────────────────────
class WorkingMemory:
    def __init__(self):
        self.notes: List[str] = []

    def add(self, note: str):
        self.notes.append(note)

    def context(self) -> str:
        if not self.notes:
            return "(empty)"
        return "\n".join(f"[{i}] {note}" for i, note in enumerate(self.notes))

    def clear(self):
        self.notes = []


wm = WorkingMemory()

wm.add("User wants REST API with FastAPI")
wm.add("Database: PostgreSQL + SQLAlchemy")
wm.add("Auth: JWT with 1-hour expiry")

print("Working memory:")
print(wm.context())
print()


# ── Episodic Memory -- timestamped interaction logs ─────────────────────────
class EpisodicMemory:
    def __init__(self):
        self.episodes = []

    def record(self, query: str, response: str):
        self.episodes.append({
            "ts": datetime.now().isoformat(),
            "q": query,
            "r": response
        })

    def recall(self, query: str, top_k: int = 3):
        q_words = set(query.lower().split())

        scored = [
            (len(q_words & set(ep["q"].lower().split())), ep)
            for ep in self.episodes
        ]

        # sort by score only
        scored.sort(key=lambda x: x[0], reverse=True)

        return [ep for _, ep in scored[:top_k]]


ep = EpisodicMemory()

ep.record("How to deploy?", "Use ECS or Lambda.")
ep.record("Which database?", "PostgreSQL for relational.")
ep.record("How to handle auth?", "JWT tokens.")

results = ep.recall("best database")

if results:
    print(f"Episodic recall for 'database': {results[0]['r']}\n")


# ── Semantic Memory -- extracted facts ──────────────────────────────────────
class SemanticMemory:
    def __init__(self):
        self.facts: Dict[str, list] = {}

    def store(self, entity: str, fact: str):
        self.facts.setdefault(entity, [])

        if fact not in self.facts[entity]:
            self.facts[entity].append(fact)

    def recall(self, entity: str):
        return self.facts.get(entity, [])

    def dump(self):
        return "\n".join(
            f"{entity}: {', '.join(facts)}"
            for entity, facts in self.facts.items()
        )


sem = SemanticMemory()

sem.store("user", "Name is Alex")
sem.store("user", "Prefers Python")
sem.store("project", "REST API with FastAPI")

print("Semantic memory:")
print(sem.dump())
print()


# ── Procedural Memory -- learned strategies ─────────────────────────────────
class ProceduralMemory:
    def __init__(self):
        self.procedures: Dict[str, list] = {}

    def learn(self, category: str, steps: list):
        self.procedures[category] = steps

    def recall(self, category: str):
        return self.procedures.get(category, [])


proc = ProceduralMemory()

proc.learn(
    "error_handling",
    ["Wrap in try/except", "Log with context", "Return proper HTTP status"]
)

proc.learn(
    "api_design",
    ["Define OpenAPI schema", "Version the API", "Add rate limiting"]
)

print("Procedural 'error_handling':")
print(proc.recall("error_handling"))

Buffer (max 4, added 12): 4 items
Contents: ['Message 5', 'Reply 5', 'Message 6', 'Reply 6']
Oldest messages automatically dropped.

Working memory:
[0] User wants REST API with FastAPI
[1] Database: PostgreSQL + SQLAlchemy
[2] Auth: JWT with 1-hour expiry

Episodic recall for 'database': Use ECS or Lambda.

Semantic memory:
user: Name is Alex, Prefers Python
project: REST API with FastAPI

Procedural 'error_handling':
['Wrap in try/except', 'Log with context', 'Return proper HTTP status']


In [ ]:
# -- Unified Memory Agent (all types combined) --
class UnifiedAgent:
    def __init__(self, system_prompt: str):
        self.system = system_prompt
        self.buffer = BufferMemory(max_size=10)
        self.working = WorkingMemory()
        self.episodic = EpisodicMemory()
        self.semantic = SemanticMemory()
        self.procedural = ProceduralMemory()

    def _build_context(self) -> str:
        parts = [self.system]
        facts = self.semantic.dump()
        if facts: parts.append(f"Known facts:\n{facts}")
        if self.procedural.procedures:
            p = "\n".join(f"  {k}: {v}" for k, v in self.procedural.procedures.items())
            parts.append(f"Procedures:\n{p}")
        wm = self.working.context()
        if "(empty)" not in wm: parts.append(f"Working memory:\n{wm}")
        return "\n\n".join(parts)

    def chat(self, user_input: str) -> str:
        self.buffer.add("user", user_input)

        # Check episodic memory
        past = self.episodic.recall(user_input, top_k=2)
        ep_ctx = ""
        if past:
            ep_ctx = "\nPast interactions:\n" + "\n".join(
                f"  Q: {e['q'][:80]} -> A: {e['r'][:80]}" for e in past)

        ctx = self._build_context() + ep_ctx
        messages = [{"role": "system", "content": ctx}] + self.buffer.get_messages()
        resp = call_llm_messages(messages, temperature=0.7)

        self.buffer.add("assistant", resp)
        self.episodic.record(user_input, resp)

        # Auto-extract facts from statements
        if any(kw in user_input.lower() for kw in ["my name", "i prefer", "we use", "our stack"]):
            raw = call_llm(
                f"Extract facts from: '{user_input}'. Return JSON: {{\"entity\": [\"fact\"]}}",
                temperature=0)
            try:
                clean = raw.strip().removeprefix("```json").removeprefix("```").removesuffix("```").strip()
                for ent, facts in json.loads(clean).items():
                    for f in facts: self.semantic.store(ent, f)
            except: pass
        return resp

# Demo
agent = UnifiedAgent("We are a technical assistant with long-term memory.")
agent.semantic.store("user", "Name is Alex")
agent.procedural.learn("debugging", ["Check logs", "Reproduce locally", "Add breakpoints"])

print("UNIFIED MEMORY AGENT")
print("=" * 50)
for msg in [
    "We use Python and FastAPI for our APIs.",
    "What database would you recommend?",
    "What do you remember about me and our project?",
]:
    print(f"\n[USER] {msg}")
    resp = agent.chat(msg)
    print(f"[AGENT] {resp[:250]}")

print(f"\n--- Memory State ---")
print(f"Buffer: {len(agent.buffer)} msgs | Episodic: {len(agent.episodic.episodes)} eps")
print(f"Semantic:\n{agent.semantic.dump()}")
print(f"Procedural: {list(agent.procedural.procedures.keys())}")

UNIFIED MEMORY AGENT

[USER] We use Python and FastAPI for our APIs.
[AGENT] Great! FastAPI is an excellent choice for building APIs in Python due to its speed, ease of use, and support for asynchronous programming. If you have any questions or need assistance related to FastAPI or Python, feel free to ask!

[USER] What database would you recommend?
[AGENT] The choice of a database depends on your specific use case, but here are a few popular options that pair well with Python and FastAPI:

1. **PostgreSQL**: A powerful, open-source relational database with strong community support. It's known for its r

[USER] What do you remember about me and our project?
[AGENT] You are Alex, and you use Python and FastAPI for your APIs. If there's anything else you'd like to share or if there are specific areas of your project you'd like to discuss, feel free to let me know!

--- Memory State ---
Buffer: 6 msgs | Episodic: 3 eps
Semantic:
user: Name is Alex
Python: We use Python for our APIs.
FastA

---
# Summary


| Component | What We Built |
|-----------|--------------|
| **Agent loop** | `while True` + LLM + tool execution |
| **Tool system** | Registry, JSON-based calling, execution |
| **ReAct** | Thought-Action-Observation interleaving |
| **Reflexion** | Self-evaluation and iterative improvement |
| **Plan-and-Execute** | Separate planning from execution |
| **Multi-agent** | Parallel, supervisor, and debate patterns |
| **Context engineering** | Loadout, pruning, summarization, offloading |
| **Memory** | Buffer, working, episodic, semantic, procedural |